# 03 — Evaluation

Compute and summarise the four metrics for every generated image.

| Metric | What it measures | Better when |
|--------|-----------------|-------------|
| `clip_score` | Cosine similarity × 100 between the image and its positive prompt (ViT-B-32) | Higher |
| `lpips` | Mean pairwise LPIPS across 3 seeds in a (scene, strategy, mode) group — perceptual consistency | Lower |
| `diversity` | Mean pairwise (1 − cosine) of CLIP image embeddings across seeds — visual variety | Higher |
| `aesthetic` | LAION aesthetic predictor score 1–10 (falls back to sharpness + colorfulness proxy) | Higher |

**VRAM budget:** CLIP ViT-B-32 (~600 MB) + LPIPS AlexNet (~50 MB) + aesthetic MLP (<1 MB) ≈ 0.7 GB loaded.
Peak during batch encoding stays under 2 GB — safe to run alongside other Colab processes.

---

## Cell 1 — Environment Setup

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

from google.colab import drive
import yaml

drive.mount("/content/drive", force_remount=False)

DRIVE_ROOT = Path("/content/drive/MyDrive/ikea-sd")
REPO_DIR   = Path("/content/ikea-sd")

# Pull latest code if repo already exists, otherwise clone
if REPO_DIR.is_dir():
    os.system(f"git -C {REPO_DIR} pull --ff-only")
else:
    # ── EDIT: replace with your repo URL ──────────────────────────────────
    REPO_URL = "https://github.com/YOUR_USERNAME/ikea-sd.git"
    # ──────────────────────────────────────────────────────────────────────
    ret = os.system(f"git clone {REPO_URL} {REPO_DIR}")
    if ret != 0:
        raise RuntimeError(f"git clone failed (exit {ret}).")

%cd /content/ikea-sd

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

# Install dependencies — torch is already present on Colab T4
with open("requirements.txt") as f:
    reqs = [
        ln.strip()
        for ln in f
        if ln.strip() and not ln.startswith("#") and not ln.lower().startswith("torch")
    ]

print(f"Installing {len(reqs)} packages…")
result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", *reqs],
    capture_output=True, text=True,
)
if result.returncode != 0:
    print("STDERR:", result.stderr[-2000:])
    raise RuntimeError("pip install failed.")

print("Environment ready.")

# ── Load config and patch paths for Colab layout ─────────────────────────
with open(REPO_DIR / "config.yaml") as f:
    cfg = yaml.safe_load(f)

cfg["paths"]["data_raw"]       = str(REPO_DIR / "data" / "raw")
cfg["paths"]["data_processed"] = str(REPO_DIR / "data" / "processed")
cfg["paths"]["outputs"]        = str(REPO_DIR / "outputs" / "generated_images")
cfg["paths"]["db_path"]        = str(DRIVE_ROOT / "results.db")

# Sync generated images from Drive to local so evaluator can open them
LOCAL_IMGS  = REPO_DIR / "outputs" / "generated_images"
DRIVE_IMGS  = DRIVE_ROOT / "outputs" / "generated_images"
LOCAL_IMGS.mkdir(parents=True, exist_ok=True)

if DRIVE_IMGS.is_dir():
    print("Syncing generated images from Drive → local…")
    !rsync -a --progress /content/drive/MyDrive/ikea-sd/outputs/generated_images/ \
        /content/ikea-sd/outputs/generated_images/
    n = len(list(LOCAL_IMGS.rglob("*.png")))
    print(f"  {n} images available locally.")
else:
    print(f"WARNING: {DRIVE_IMGS} not found — run notebook 02 first.")

# Verify GPU for faster CLIP encoding
import torch
print(f"\nCUDA available : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device         : {torch.cuda.get_device_name(0)}")

## Cell 2 — Compute All Metrics

Loads CLIP ViT-B-32, LPIPS AlexNet, and the LAION aesthetic MLP once, then
batch-processes every run in the database.  Already-scored runs are skipped
automatically (`force=False`).  Pass `force=True` to re-score everything.

In [ ]:
import logging
from pathlib import Path

import torch

from src.evaluator import evaluate_all

logging.basicConfig(
    level=logging.INFO,
    format="%(levelname)s  %(name)s  %(message)s",
    force=True,
)

# ── VRAM snapshot before loading evaluation models ────────────────────────
def _vram_gb(kind: str = "reserved") -> float:
    if not torch.cuda.is_available():
        return 0.0
    fn = torch.cuda.memory_reserved if kind == "reserved" else torch.cuda.memory_allocated
    return fn(0) / 1024**3

print(f"VRAM before evaluation models : {_vram_gb():.2f} GB")

# ── Run evaluation ────────────────────────────────────────────────────────
OUT_CSV = Path("/content/ikea-sd/outputs/metrics.csv")
OUT_CSV.parent.mkdir(parents=True, exist_ok=True)

metrics_df = evaluate_all(
    db_path=cfg["paths"]["db_path"],
    config=cfg,
    force=False,       # set True to recompute already-scored runs
    out_csv=str(OUT_CSV),
)

print(f"\nVRAM after evaluation        : {_vram_gb():.2f} GB")
print(f"Rows in metrics DataFrame    : {len(metrics_df)}")
print(f"Columns                      : {list(metrics_df.columns)}")
print(f"\nFirst 3 rows:")
display(metrics_df[["run_id", "scene_id", "strategy", "control_mode",
                     "clip_score", "lpips", "diversity", "aesthetic"]].head(3))

## Cell 3 — Load metrics.csv into pandas

In [ ]:
from pathlib import Path
import pandas as pd

OUT_CSV = Path("/content/ikea-sd/outputs/metrics.csv")

if not OUT_CSV.is_file():
    raise FileNotFoundError(
        f"{OUT_CSV} not found.  Re-run Cell 2 to generate it."
    )

df = pd.read_csv(OUT_CSV)

METRICS = ["clip_score", "lpips", "diversity", "aesthetic"]
GROUPS  = ["strategy", "control_mode"]

# Validate expected columns are present
missing = [c for c in METRICS + GROUPS if c not in df.columns]
if missing:
    raise ValueError(f"metrics.csv is missing columns: {missing}")

print(f"Loaded {len(df)} rows from {OUT_CSV}")
print(f"\nData types:\n{df[METRICS].dtypes.to_string()}")
print(f"\nMissing values per metric column:")
print(df[METRICS].isna().sum().to_string())
print(f"\nStrategies   : {sorted(df['strategy'].unique())}")
print(f"Control modes: {sorted(df['control_mode'].unique())}")
print(f"Seeds        : {sorted(df['seed'].unique())}")
print(f"Unique scenes: {df['scene_id'].nunique()}")

## Cell 4 — Summary Table: Mean ± Std per (strategy, control_mode)

In [ ]:
import pandas as pd
import numpy as np

METRICS       = ["clip_score", "lpips", "diversity", "aesthetic"]
GROUPS        = ["strategy", "control_mode"]
# Which direction is "better" for each metric — used for highlighting
HIGHER_BETTER = {"clip_score": True, "lpips": False, "diversity": True, "aesthetic": True}

# ── Build mean ± std aggregation ──────────────────────────────────────────
agg = df.groupby(GROUPS)[METRICS].agg(["mean", "std"]).round(4)

# Flatten MultiIndex columns: (metric, stat) → "metric_mean" / "metric_std"
agg.columns = [f"{metric}_{stat}" for metric, stat in agg.columns]
agg = agg.reset_index()

# ── Add a "mean ± std" display column for each metric ─────────────────────
for m in METRICS:
    mean_col = f"{m}_mean"
    std_col  = f"{m}_std"
    agg[m]   = agg.apply(
        lambda r: f"{r[mean_col]:.3f} ± {r[std_col]:.3f}"
        if pd.notna(r[std_col]) else f"{r[mean_col]:.3f}",
        axis=1,
    )

display_cols = GROUPS + METRICS
summary = agg[display_cols].copy()

# ── Highlight best value per metric (green = best direction) ──────────────
def _highlight_best(col_data: pd.Series) -> list[str]:
    """Return CSS background-color styles; highlight the optimal row."""
    metric_name = col_data.name
    if metric_name not in HIGHER_BETTER:
        return [""] * len(col_data)
    # Extract the mean values from "mean ± std" strings
    means = col_data.str.split(" ").str[0].astype(float)
    best_idx = means.idxmax() if HIGHER_BETTER[metric_name] else means.idxmin()
    return [
        "background-color: #c8f0c8; font-weight: bold" if i == best_idx else ""
        for i in col_data.index
    ]

styled = (
    summary.style
    .apply(_highlight_best, subset=METRICS)
    .set_caption("Mean ± Std per (strategy, control_mode) — green = best")
    .set_table_styles([
        {"selector": "caption", "props": [("font-size", "13px"), ("font-weight", "bold")]},
        {"selector": "th",      "props": [("text-align", "center")]},
        {"selector": "td",      "props": [("text-align", "center"), ("padding", "4px 10px")]},
    ])
)

print("Summary table (strategy × control_mode):")
display(styled)

# Also print a plain-text version for quick scanning
print("\nPlain text (sorted by clip_score_mean ↓):")
plain = agg[GROUPS + [f"{m}_mean" for m in METRICS]].copy()
plain.columns = GROUPS + [f"{m}" for m in METRICS]
plain = plain.sort_values("clip_score", ascending=False)
print(plain.to_string(index=False, float_format="{:.4f}".format))

## Cell 5 — Save Summary Table to report/figures/

In [ ]:
from pathlib import Path
import shutil

FIGURES_DIR = Path("/content/ikea-sd/report/figures")
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# ── 1. Save the human-readable "mean ± std" summary ──────────────────────
SUMMARY_CSV = FIGURES_DIR / "summary_table.csv"
summary.to_csv(SUMMARY_CSV, index=False)
print(f"Saved summary table  → {SUMMARY_CSV}")

# ── 2. Save the numeric (mean / std separated) version for programmatic use
NUMERIC_CSV = FIGURES_DIR / "summary_table_numeric.csv"
numeric_cols = GROUPS + [f"{m}_mean" for m in METRICS] + [f"{m}_std" for m in METRICS]
agg[numeric_cols].to_csv(NUMERIC_CSV, index=False)
print(f"Saved numeric table  → {NUMERIC_CSV}")

# ── 3. Persist both to Drive so they survive session restarts ─────────────
DRIVE_FIGURES = Path("/content/drive/MyDrive/ikea-sd/report/figures")
DRIVE_FIGURES.mkdir(parents=True, exist_ok=True)

for src in (SUMMARY_CSV, NUMERIC_CSV):
    dst = DRIVE_FIGURES / src.name
    shutil.copy2(src, dst)
    print(f"Backed up            → {dst}")

# ── 4. Quick sanity print ─────────────────────────────────────────────────
print(f"\nContents of {FIGURES_DIR}:")
for p in sorted(FIGURES_DIR.iterdir()):
    print(f"  {p.name}  ({p.stat().st_size / 1024:.1f} KB)")